<a href="https://colab.research.google.com/github/sw030701-ai/motor-control-optimization/blob/main/experiments/02_pid_baseline_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 · Baseline PID Tuning Experiment
### Fixed Nominal Motor Plant → Sequential Manual PID Tuning → Baseline Record

---

### Overview

이 notebook은 `01_dc_motor_model.ipynb`에서 정한 nominal motor plant와 reference speed를 사용해 **Conventional PID Baseline**을 만든다.

```text
Fixed DC Motor Plant
      ↓
Fixed Step Reference Speed
      ↓
Kp Scan
      ↓
Ki Scan
      ↓
Small Kd Scan
      ↓
Baseline PID Gains Fix
      ↓
J_baseline 계산 및 record 저장
```

여기서는 motor parameter 선정과 open-loop validation을 반복하지 않는다. 그 내용은 `01_dc_motor_model.ipynb`가 담당한다.

In [1]:
import os, sys, json, platform, subprocess, math
from pathlib import Path


def _in_colab():
    return "google.colab" in sys.modules


def _find_root(start: Path) -> Path:
    p = start.resolve()
    for cand in [p, *p.parents]:
        if (cand / "src").exists() and (cand / "docs").exists():
            return cand
    return p


REPO_URL = "https://github.com/sw030701-ai/motor-control-optimization.git"

if _in_colab():
    root = Path("/content/motor-control-optimization")
    if not root.exists():
        subprocess.run(["git", "clone", REPO_URL, str(root)], check=True)
    else:
        subprocess.run(["git", "pull", "--ff-only"], cwd=root, check=False)
    ROOT = root
else:
    ROOT = _find_root(Path.cwd())

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

os.environ.setdefault("MPLCONFIGDIR", "/tmp/mplconfig")
os.environ.setdefault("XDG_CACHE_HOME", "/tmp/xdgcache")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["XDG_CACHE_HOME"]).mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib
if not _in_colab():
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams["axes.unicode_minus"] = False


def _git(*args):
    try:
        return subprocess.check_output(["git", *args], cwd=ROOT, text=True).strip()
    except Exception:
        return None

ENV = {
    "root": str(ROOT),
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "git_commit": _git("rev-parse", "HEAD"),
    "git_dirty": bool(_git("status", "--short")),
}

print(json.dumps(ENV, indent=2, ensure_ascii=False))

{
  "root": "/Users/seong-uuimac/Documents/Codex/2026-09-06/referenced-chatgpt-conversation-this-is-an/repo",
  "python": "3.13.9",
  "numpy": "2.3.5",
  "pandas": "2.3.3",
  "git_commit": "94987e2b258908c4477adee8d4a1e26da4faf6ae",
  "git_dirty": true
}


In [2]:
SAVE_ARTIFACTS = True
SHOW_SCAN_TABLES = True

RESULT_TABLE_DIR = Path("results") / "tables"
RESULT_FIGURE_DIR = Path("results") / "figures"
if SAVE_ARTIFACTS:
    RESULT_TABLE_DIR.mkdir(parents=True, exist_ok=True)
    RESULT_FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("SAVE_ARTIFACTS:", SAVE_ARTIFACTS)
print("SHOW_SCAN_TABLES:", SHOW_SCAN_TABLES)

SAVE_ARTIFACTS: True
SHOW_SCAN_TABLES: True


## Section 1 — Load Fixed Plant and Reference

- **Plant**: `src.motor.dc_motor.nominal_dc_motor_params()`에 정의된 literature-based nominal motor.
- **Reference**: `01_dc_motor_model.ipynb`에서 정한 reachable step reference.

만약 `01` 결과 파일이 아직 없으면, 같은 공식으로 reference를 다시 계산한다.

In [3]:
from src.motor.dc_motor import NOMINAL_MOTOR_SOURCE, nominal_dc_motor_params
from src.simulation.pid_simulation import reference_from_reachable_speed

params = nominal_dc_motor_params()
V_MAX = 12.0
REFERENCE_FILE = RESULT_TABLE_DIR / "reference_selection_record.json"

if REFERENCE_FILE.exists():
    reference_record = json.loads(REFERENCE_FILE.read_text(encoding="utf-8"))
    OMEGA_REF = float(reference_record["omega_ref_rad_s"])
else:
    reference_record = {
        "V_max": V_MAX,
        "omega_ss_max_rad_s": params.no_load_steady_state_speed(V_MAX),
        "reference_fraction": 0.50,
        "omega_ref_rad_s": round(reference_from_reachable_speed(params, V_MAX, fraction=0.50), 2),
        "note": "Recomputed because 01 reference record was not found.",
    }
    OMEGA_REF = float(reference_record["omega_ref_rad_s"])

plant_summary = pd.DataFrame([{
    "R": params.R,
    "L": params.L,
    "J_m": params.J_m,
    "b": params.b,
    "K_t": params.K_t,
    "K_e": params.K_e,
    "V_max": V_MAX,
    "omega_ref": OMEGA_REF,
}])

display(plant_summary)
print(json.dumps(reference_record, indent=2, ensure_ascii=False))

,R,L,J_m,b,K_t,K_e,V_max,omega_ref
0,0.18644,0.0063,0.013767,0.049813,0.020375,0.020375,12.0,12.6


{
  "V_max": 12.0,
  "omega_ss_max_rad_s": 25.200271699744086,
  "reference_fraction": 0.5,
  "omega_ref_rad_s": 12.6,
  "simulation_time_s": 10.0,
  "dt_s": 0.001
}


## Section 2 — Cost Function and v1 Acceptance Criteria

Baseline tuning은 optimizer가 아니다. 여기서는 사람이 manual tuning할 때의 순서를 reproducible하게 기록한다.

Cost는 baseline을 선택한 뒤 benchmark로 계산한다.

$$
\mathcal{J}=0.60J_{tracking}+0.25J_{overshoot}+0.15J_{control}
$$

이 weight는 v1 design choice이며, optimizer가 바꾸는 변수는 아니다.

아래 acceptance criteria는 constrained optimization에서도 같은 feasibility constraints로 사용한다.

In [4]:
from src.optimization.cost_function import V1_CONSTRAINTS

COST_WEIGHTS = {
    "tracking": 0.60,
    "overshoot": 0.25,
    "control": 0.15,
}

ACCEPTANCE = {
    "overshoot_percent_max": 100.0 * V1_CONSTRAINTS.overshoot_limit,
    "steady_state_error_percent_max": 100.0 * V1_CONSTRAINTS.steady_state_error_limit,
    "settling_time_s_max": V1_CONSTRAINTS.settling_time_limit,
    "saturation_percent_max_exclusive": 100.0 * V1_CONSTRAINTS.saturation_fraction_limit,
    "stable_no_divergence": True,
}

print("Cost weights:")
print(json.dumps(COST_WEIGHTS, indent=2, ensure_ascii=False))
print("\nAcceptance criteria:")
print(json.dumps(ACCEPTANCE, indent=2, ensure_ascii=False))

Cost weights:
{
  "tracking": 0.6,
  "overshoot": 0.25,
  "control": 0.15
}

Acceptance criteria:
{
  "overshoot_percent_max": 10.0,
  "steady_state_error_percent_max": 2.0,
  "settling_time_s_max": 2.0,
  "saturation_percent_max_exclusive": 5.0,
  "stable_no_divergence": true
}


## Section 3 — Sequential Manual PID Tuning

Manual baseline tuning 순서는 다음과 같다.

```text
1. Ki = 0, Kd = 0으로 두고 Kp scan
2. Kp를 고정하고 Ki scan
3. 필요하면 작은 Kd scan
4. v1 acceptance criteria를 만족하면 baseline gains 고정
```

이 단계에서 여러 gain 후보를 넣고, 각 후보마다 closed-loop response를 simulation해서 표의 metric을 계산한다.

In [5]:
from src.simulation.baseline_tuning import BaselineTuningConfig, sequential_baseline_tuning

config = BaselineTuningConfig(
    omega_ref=OMEGA_REF,
    V_max=V_MAX,
    simulation_time=10.0,
    dt=0.001,
    settling_target=2.0,
)

tuning = sequential_baseline_tuning(params, config)

kp_scan = pd.DataFrame(tuning["Kp_scan"])
ki_scan = pd.DataFrame(tuning["Ki_scan"])
kd_scan = pd.DataFrame(tuning["Kd_scan"])
selected = tuning["selected"]
final_gains = tuning["final_gains"]

columns = [
    "K_p", "K_i", "K_d", "total", "tracking", "overshoot", "overshoot_cost", "control",
    "overshoot_percent", "settling_time", "steady_state_error_percent",
    "voltage_max_abs", "saturation_percent", "omega_final",
]

print("Selected baseline gains:")
print(final_gains)

Selected baseline gains:
PIDGains(K_p=0.8, K_i=2.0, K_d=0.002)


In [6]:
if SHOW_SCAN_TABLES:
    print("Kp scan")
    display(kp_scan[columns].round(6))

Kp scan


,K_p,K_i,K_d,total,tracking,overshoot,overshoot_cost,control,overshoot_percent,settling_time,steady_state_error_percent,voltage_max_abs,saturation_percent,omega_final
0,0.02,0.0,0.0,0.276398,0.460561,0.0,0.0,0.000407,0.0,inf,95.969248,0.252,0.0,0.507875
1,0.04,0.0,0.0,0.255591,0.425607,0.0,0.0,0.001508,0.0,inf,92.250845,0.504,0.0,0.976393
2,0.06,0.0,0.0,0.237163,0.394483,0.0,0.0,0.003152,0.0,inf,88.809840,0.756,0.0,1.409960
3,0.08,0.0,0.0,0.220773,0.366651,0.0,0.0,0.005218,0.0,inf,85.616306,1.008,0.0,1.812345
4,0.10,0.0,0.0,0.206138,0.341661,0.0,0.0,0.007612,0.0,inf,82.644473,1.260,0.0,2.186796
5,0.15,0.0,0.0,0.175773,0.289313,0.0,0.0,0.014567,0.0,inf,76.045431,1.890,0.0,3.018276
6,0.20,0.0,0.0,0.152223,0.248130,0.0,0.0,0.022300,0.0,inf,70.422311,2.520,0.0,3.726789
7,0.30,0.0,0.0,0.118752,0.188331,0.0,0.0,0.038359,0.0,inf,61.349438,3.780,0.0,4.869971
8,0.50,0.0,0.0,0.081674,0.119072,0.0,0.0,0.068208,0.0,inf,48.780218,6.300,0.0,6.453692
9,0.80,0.0,0.0,0.057376,0.069668,0.0,0.0,0.103835,0.0,inf,37.313181,10.080,0.0,7.898539


**Kp 선택 기준 요약**

`K_i = 0`, `K_d = 0`으로 고정한 P-only scan에서 다음 조건을 처음 만족하는 값을 고른다.

```text
ω_final >= 0.60 × ω_ref
overshoot <= 5%
saturation_percent < 1%
```

In [7]:
if SHOW_SCAN_TABLES:
    print("Ki scan")
    display(ki_scan[columns].round(6))

Ki scan


,K_p,K_i,K_d,total,tracking,overshoot,overshoot_cost,control,overshoot_percent,settling_time,steady_state_error_percent,voltage_max_abs,saturation_percent,omega_final
0,0.8,0.05,0.0,0.044096,0.041730,0.0,0.0,0.127051,0.0,inf,25.597759,10.080630,0.0,9.437725
1,0.8,0.10,0.0,0.037557,0.025850,0.0,0.0,0.146979,0.0,inf,17.503572,10.081583,0.0,10.480570
2,0.8,0.20,0.0,0.033245,0.011087,0.0,0.0,0.177285,0.0,inf,8.102069,10.084103,0.0,11.658399
3,0.8,0.40,0.0,0.033602,0.003140,0.0,0.0,0.211452,0.0,8.920,1.663364,10.091900,0.0,12.422632
4,0.8,0.80,0.0,0.035965,0.000728,0.0,0.0,0.236852,0.0,4.233,0.057801,10.118477,0.0,12.594905
5,0.8,1.20,0.0,0.037110,0.000307,0.0,0.0,0.246176,0.0,2.653,0.001470,10.160954,0.0,12.599896
6,0.8,2.00,0.0,0.038183,0.000113,0.0,0.0,0.254103,0.0,1.335,0.000000,10.298972,0.0,12.600000


**Ki 선택 기준 요약**

`K_p`를 고정한 뒤 `K_i`를 증가시켜 steady-state error를 제거한다.

```text
steady-state error <= 2%
settling time <= 2.0 s
overshoot <= 10%
persistent saturation 없음
```

In [8]:
if SHOW_SCAN_TABLES:
    print("Kd scan")
    display(kd_scan[columns].round(6))

Kd scan


,K_p,K_i,K_d,total,tracking,overshoot,overshoot_cost,control,overshoot_percent,settling_time,steady_state_error_percent,voltage_max_abs,saturation_percent,omega_final
0,0.8,2.0,0.0000,0.038183,0.000113,0.0,0.0,0.254103,0.0,1.335,0.0,10.298972,0.0,12.6
1,0.8,2.0,0.0001,0.038183,0.000113,0.0,0.0,0.254100,0.0,1.334,0.0,10.295956,0.0,12.6
2,0.8,2.0,0.0002,0.038183,0.000113,0.0,0.0,0.254098,0.0,1.333,0.0,10.292941,0.0,12.6
3,0.8,2.0,0.0005,0.038182,0.000114,0.0,0.0,0.254090,0.0,1.331,0.0,10.284282,0.0,12.6
4,0.8,2.0,0.0010,0.038180,0.000114,0.0,0.0,0.254078,0.0,1.328,0.0,10.269915,0.0,12.6
5,0.8,2.0,0.0020,0.038177,0.000114,0.0,0.0,0.254054,0.0,1.322,0.0,10.242803,0.0,12.6


**Kd 선택 기준 요약**

`K_d`는 꼭 큰 값을 쓸 필요가 없다. v1에서는 안정 조건을 만족하는 후보 중 settling time과 overshoot가 좋은 작은 derivative gain을 선택한다.

## Section 4 — Baseline Closed-Loop Simulation

선택된 baseline gains를 고정하고 closed-loop response를 다시 계산한다.

```text
ω_ref(t)
      ↓
e(t) = ω_ref(t) - ω(t)
      ↓
PID Controller
      ↓
V(t)
      ↓
DC Motor Plant
      ↓
ω(t)
```

In [9]:
from src.optimization.cost_function import accepted_baseline, compute_cost
from src.simulation.pid_simulation import simulate_pid

baseline_result = simulate_pid(
    motor_params=params,
    gains=final_gains,
    omega_ref=OMEGA_REF,
    V_max=V_MAX,
    simulation_time=config.simulation_time,
    dt=config.dt,
)

baseline_cost = compute_cost(baseline_result, omega_ref=OMEGA_REF, V_max=V_MAX)

baseline_record = pd.DataFrame([{
    "K_p_baseline": final_gains.K_p,
    "K_i_baseline": final_gains.K_i,
    "K_d_baseline": final_gains.K_d,
    "reference_speed_rad_s": OMEGA_REF,
    "V_max": V_MAX,
    "J_baseline": baseline_cost["total"],
    "J_tracking": baseline_cost["tracking"],
    "J_overshoot": baseline_cost["overshoot_cost"],
    "J_control": baseline_cost["control"],
    "overshoot_percent": baseline_cost["overshoot_percent"],
    "steady_state_error_percent": baseline_cost["steady_state_error_percent"],
    "settling_time_s": baseline_cost["settling_time"],
    "max_abs_voltage": baseline_cost["voltage_max_abs"],
    "saturation_percent": baseline_cost["saturation_percent"],
    "accepted_v1": accepted_baseline(baseline_cost),
}])

display(baseline_record.T.rename(columns={0: "value"}))

,value
K_p_baseline,0.8
K_i_baseline,2.0
K_d_baseline,0.002
reference_speed_rad_s,12.6
V_max,12.0
J_baseline,0.038177
J_tracking,0.000114
J_overshoot,0.0
J_control,0.254054
overshoot_percent,0.0


In [10]:
fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)

axes[0].plot(baseline_result["time"], baseline_result["omega"], label="Baseline PID speed")
axes[0].axhline(OMEGA_REF, linestyle="--", color="tab:red", label="Reference")
axes[0].set_ylabel("omega [rad/s]")
axes[0].set_title("Baseline PID Closed-Loop Response")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(baseline_result["time"], baseline_result["error"], color="tab:orange")
axes[1].axhline(0.0, linestyle="--", color="black", linewidth=1)
axes[1].set_ylabel("error [rad/s]")
axes[1].grid(True, alpha=0.3)

axes[2].plot(baseline_result["time"], baseline_result["voltage"], color="tab:green")
axes[2].axhline(V_MAX, linestyle="--", color="tab:red", linewidth=1)
axes[2].axhline(-V_MAX, linestyle="--", color="tab:red", linewidth=1)
axes[2].set_xlabel("Time [s]")
axes[2].set_ylabel("voltage [V]")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
if SAVE_ARTIFACTS:
    plt.savefig(RESULT_FIGURE_DIR / "baseline_pid_response.png", dpi=160)
plt.show()

/var/folders/js/s5xbjpb56mn1ysk430xc81w40000gn/T/ipykernel_45037/4010996759.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Section 5 — Tuning Record Export

Baseline tuning record를 `results/tables/`에 저장한다.

```text
baseline_pid_tuning_record.csv
baseline_pid_tuning_record.md
baseline_pid_tuning_record.json
baseline_kp_scan.csv
baseline_ki_scan.csv
baseline_kd_scan.csv
```

In [11]:
record = baseline_record.iloc[0].to_dict()
record["motor_parameters"] = {
    "R": params.R,
    "L": params.L,
    "J_m": params.J_m,
    "b": params.b,
    "K_t": params.K_t,
    "K_e": params.K_e,
}
record["motor_source"] = NOMINAL_MOTOR_SOURCE
record["environment"] = ENV

if SAVE_ARTIFACTS:
    baseline_record.to_csv(RESULT_TABLE_DIR / "baseline_pid_tuning_record.csv", index=False)
    md_rows = ["| Item | Value |", "|---|---:|"]
    for key, value in baseline_record.iloc[0].items():
        if isinstance(value, float):
            text = f"{value:.10g}"
        else:
            text = str(value)
        md_rows.append(f"| `{key}` | {text} |")
    (RESULT_TABLE_DIR / "baseline_pid_tuning_record.md").write_text(
        "\n".join(md_rows) + "\n",
        encoding="utf-8",
    )
    with open(RESULT_TABLE_DIR / "baseline_pid_tuning_record.json", "w", encoding="utf-8") as f:
        json.dump(record, f, indent=2, ensure_ascii=False)
    kp_scan.to_csv(RESULT_TABLE_DIR / "baseline_kp_scan.csv", index=False)
    ki_scan.to_csv(RESULT_TABLE_DIR / "baseline_ki_scan.csv", index=False)
    kd_scan.to_csv(RESULT_TABLE_DIR / "baseline_kd_scan.csv", index=False)

print("Baseline tuning record saved." if SAVE_ARTIFACTS else "SAVE_ARTIFACTS=False, no files saved.")
print(json.dumps({
    "K_p": final_gains.K_p,
    "K_i": final_gains.K_i,
    "K_d": final_gains.K_d,
    "J_baseline": baseline_cost["total"],
    "overshoot_percent": baseline_cost["overshoot_percent"],
    "steady_state_error_percent": baseline_cost["steady_state_error_percent"],
    "settling_time_s": baseline_cost["settling_time"],
    "accepted_v1": accepted_baseline(baseline_cost),
}, indent=2, ensure_ascii=False))

Baseline tuning record saved.
{
  "K_p": 0.8,
  "K_i": 2.0,
  "K_d": 0.002,
  "J_baseline": 0.03817651842324221,
  "overshoot_percent": 0.0,
  "steady_state_error_percent": 2.158820987935383e-07,
  "settling_time_s": 1.322,
  "accepted_v1": true
}


## Final Summary

이번 notebook에서 baseline PID gains는 다음과 같이 고정한다.

```text
K_p_baseline = 0.80
K_i_baseline = 2.00
K_d_baseline = 0.002
```

이 값은 `Manual PID → Random Search → Bayesian Optimization → RL Controller` 비교에서 conventional PID benchmark로 사용한다.

> **주의**: 이 baseline은 intentionally poor baseline이 아니다. 안정적이고 납득 가능한 conventional PID를 기준점으로 세워야 이후 optimization 성능 비교가 왜곡되지 않는다.